In [93]:
# pip install executorch
# uv add executorch

In [ ]:
model = PlateClassifier()  # mobilenetv3_small_100, num_features=1024
model.load_state_dict(torch.load('model.pt', map_location='cpu'))

In [ ]:
# export friendly
class PlateModuleExportable(torch.nn.Module):
    def __init__(self, plate_module):
        super().__init__()
        self.yolo = plate_module.yolo
        self.model = plate_module.model
        self.register_buffer('mean', plate_module.mean)
        self.register_buffer('std', plate_module.std)

    def forward(self, x):  # [1, 3, 640, 640]
        det = self.yolo(x / 255.0)

        conf = det[:, :, 4:5]
        best_idx = conf.argmax(dim=1, keepdim=True)
        best_idx_expanded = best_idx.expand(-1, -1, 6)
        best_det = torch.gather(det, 1, best_idx_expanded).squeeze(1)  # [1, 6]

        x1, y1 = best_det[:, 0:1], best_det[:, 1:2]
        x2, y2 = best_det[:, 2:3], best_det[:, 3:4]

        # Normalizuj koordynaty do [-1, 1] dla grid_sample
        h, w = x.shape[2], x.shape[3]
        x1_n = 2.0 * x1 / w - 1.0
        y1_n = 2.0 * y1 / h - 1.0
        x2_n = 2.0 * x2 / w - 1.0
        y2_n = 2.0 * y2 / h - 1.0

        # Siatka 224x224 w znormalizowanej przestrzeni
        grid_x = torch.linspace(0, 1, 224, device=x.device).view(1, 1, 224, 1)
        grid_y = torch.linspace(0, 1, 224, device=x.device).view(1, 224, 1, 1)

        sample_x = x1_n + (x2_n - x1_n) * grid_x  # [1, 1, 224, 1]
        sample_y = y1_n + (y2_n - y1_n) * grid_y  # [1, 224, 1, 1]

        grid = torch.cat([
            sample_x.expand(-1, 224, -1, -1),  # [1, 224, 224, 1]
            sample_y.expand(-1, -1, 224, -1),  # [1, 224, 224, 1]
        ], dim=-1)  # [1, 224, 224, 2]

        cropped = F.grid_sample(x, grid, mode='bilinear', align_corners=False)  # [1, 3, 224, 224]

        normalized = (cropped / 255.0 - self.mean) / self.std
        chars = self.model(normalized)
        best_box = torch.cat([x1, y1, x2, y2], dim=1)

        return best_box, chars


exportable = PlateModuleExportable(plate_module)
exportable.eval()

# Test
with torch.no_grad():
    box, chars = exportable(torch.randn(1, 3, 640, 640))
    print(f"box: {box.shape}, chars: {chars.shape}")

In [ ]:
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner


exported = torch.export.export(
    exportable,
    (torch.randn(1, 3, 640, 640),),
    strict=False,
)

program = to_edge_transform_and_lower(
    exported,
    partitioner=[XnnpackPartitioner()],
).to_executorch()

with open("model.pte", "wb") as f:
    program.write_to_file(f)


In [ ]:
# weryfikacja
from executorch.runtime import Runtime

runtime = Runtime.get()
method = runtime.load_program("model.pte").load_method("forward")
et_out = method.execute([test_input])